Environment Setup and Configuration

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pickle
from google.colab import drive

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Hardware selected: {device}")

BATCH_SIZE = 128
LR = 0.001
EPOCHS = 10
MAX_LEN = 100
STEP_SIZE = 10

print("\nMounting Drive...")
drive.mount('/content/drive')
base_path = '/content/drive/My Drive/resources/KuaiRec 2.0/data/processed/'

Hardware selected: cuda

Mounting Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#Loading the data
print("Loading Data..")
with open(base_path + 'train_sequence.pkl', 'rb') as f:
  train_sequence = pickle.load(f)

with open(base_path + 'meta.pkl', 'rb') as f:
    meta = pickle.load(f)

frozen_embeddings = np.load(base_path + 'frozen_item_embeddings.npy')
frozen_embeddings = torch.tensor(frozen_embeddings, dtype = torch.float32).to(device)

print(f"Data loaded.")
print(f"Total users: {len(train_sequence)}")
print(f"Total items: {len(frozen_embeddings)}")

Loading Data..
Data loaded.
Total users: 7176
Total items: 10719


Forgotten User

In [ ]:
max_interactions = 0
best_user_id = -1

for uid, data in train_sequence.items():
  seq_len = len(data['item_sequence'])
  if seq_len > max_interactions:
    max_interactions = seq_len
    best_user_id = uid

FORGETTEN_USER_ID = best_user_id
print(f"Forgotten User ID: {FORGETTEN_USER_ID}")

Forgotten User ID: 4247


Model Dataset

In [ ]:
class SASRecDataset(Dataset):
  def __init__(self, sequences, max_len = 100, step_size = 10):
    self.data = []
    self.max_len = max_len

    for uid in sequences:
      full_seq = sequences[uid]['item_sequence']
      seq_len = len(full_seq)

      if seq_len <= max_len + 1:
        self.data.append(full_seq)
      else:
        for i in range(0, seq_len, step_size):
          chunk = full_seq[i : i + max_len + 1]

          if len(chunk) > 5:
            self.data.append(chunk)

  def __len__(self):
    return len(self.data)

  def __getitem__(self, index):
    seq = self.data[index]

    if len(seq) > self.max_len + 1:
      seq = seq[-(self.max_len + 1):]

    seq = [x + 1 for x in seq]

    pad_len = (self.max_len + 1) - len(seq)
    seq = [0] * pad_len + seq

    seq_tensor = torch.tensor(seq, dtype = torch.long)

    return {
        'input_seq': seq_tensor[:-1],
        'target_seq': seq_tensor[1:]
    }

SASRec Model Architecture

In [ ]:
class SASRec(nn.Module):
  def __init__(
      self,
      frozen_item_vectors,
      max_seq_len = 50,
      embed_dim = 64,
      heads = 2,
      layers = 2,
      dropout = 0.1):

    super(SASRec, self).__init__()

    self.max_seq_len = max_seq_len
    self.embed_dim = embed_dim

    #padding for index 0
    pad_row = torch.zeros(1, embed_dim).to(frozen_item_vectors.device)
    full_matrix = torch.cat([pad_row, frozen_item_vectors], dim = 0)

    #frozen embeddings
    self.item_embedding = nn.Embedding.from_pretrained(full_matrix, freeze = True, padding_idx=0)

    #adapter?
    self.adapter = nn.Sequential(
        nn.Linear(embed_dim, 128),
        nn.ReLU(),
        nn.Linear(128, embed_dim)
    )

    #position embeddings
    self.pos_embedding = nn.Embedding(max_seq_len, embed_dim)

    #encoder
    self.encoder_layer = nn.TransformerEncoderLayer(
        d_model=embed_dim,
        nhead=heads,
        dim_feedforward=128,
        dropout=dropout,
        batch_first=True)
    self.transformer_encoder = nn.TransformerEncoder(self.encoder_layer, num_layers=layers)

    #normalization
    self.norm = nn.LayerNorm(embed_dim)

    num_items_total = full_matrix.size(0)
    self.prediction_head = nn.Linear(embed_dim, num_items_total)

  def forward(self, item_seq):
    seq_len = item_seq.size(1)

    #input prep
    items_emb = self.item_embedding(item_seq)
    items_emb = self.adapter(items_emb)


    positions = torch.arange(seq_len, device = item_seq.device).unsqueeze(0)
    x = items_emb + self.pos_embedding(positions)

    #causal masking
    causal_mask = torch.triu(torch.ones(seq_len, seq_len, device = item_seq.device) * float('-inf'), diagonal = 1)
    padding_mask = (item_seq == 0)

    output = self.transformer_encoder(x, mask = causal_mask, src_key_padding_mask = padding_mask)
    output = self.norm(output)

    #prediction
    logits = self.prediction_head(output)

    return logits

Model Engine

In [ ]:
def train_engine(model_name, sequences, frozen_emb_matrix):
  ds = SASRecDataset(sequences, max_len = MAX_LEN, step_size = STEP_SIZE)
  dl = DataLoader(ds, batch_size = BATCH_SIZE, shuffle = True)

  model = SASRec(
      frozen_item_vectors = frozen_emb_matrix,
      max_seq_len = MAX_LEN,
      embed_dim = 64,
      heads = 2,
      layers = 2).to(device)
  optimizer = optim.Adam(model.parameters(), lr = LR)
  criterion = nn.CrossEntropyLoss(ignore_index = 0)

  model.train()

  for epoch in range(EPOCHS):
    total_loss = 0
    steps = 0

    for batch in dl:
      optimizer.zero_grad()
      input_seq = batch['input_seq'].to(device)
      target_seq = batch['target_seq'].to(device)

      logits = model(input_seq)
      logits = logits.view(-1, logits.size(-1))
      targets = target_seq.view(-1)

      loss = criterion(logits, targets)
      loss.backward()
      optimizer.step()

      total_loss += loss.item()
      steps += 1

    print(f"Epoch: {epoch + 1}, Loss: {total_loss / steps: .4f}")

  save_path = base_path + f'{model_name}.pth'
  torch.save(model.state_dict(), save_path)
  print(f"{model_name} saved.")
  return model

Frozen=True

In [ ]:
full_sequences = train_sequence.copy()

clean_sequences = train_sequence.copy()
if FORGETTEN_USER_ID in clean_sequences:
  del clean_sequences[FORGETTEN_USER_ID]
  print(f"Forgotten user {FORGETTEN_USER_ID}'s sequence is removed.")
else:
  print(f"Forgotten user {FORGETTEN_USER_ID} not found!!")

#training
baseline_model = train_engine("baseline_model", full_sequences, frozen_embeddings)
unlearned_model = train_engine("unlearned_model", clean_sequences, frozen_embeddings)

Forgotten user 4247's sequence is removed.


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch: 1, Loss:  8.1154
Epoch: 2, Loss:  8.0700
Epoch: 3, Loss:  8.0626
Epoch: 4, Loss:  8.0569
Epoch: 5, Loss:  8.0394
Epoch: 6, Loss:  7.9834
Epoch: 7, Loss:  7.9512
Epoch: 8, Loss:  7.9355
Epoch: 9, Loss:  7.9232
Epoch: 10, Loss:  7.9142
baseline_model saved.
Epoch: 1, Loss:  8.1131
Epoch: 2, Loss:  8.0701
Epoch: 3, Loss:  8.0634
Epoch: 4, Loss:  8.0581
Epoch: 5, Loss:  8.0463
Epoch: 6, Loss:  8.0005
Epoch: 7, Loss:  7.9675
Epoch: 8, Loss:  7.9443
Epoch: 9, Loss:  7.9221
Epoch: 10, Loss:  7.8900
unlearned_model saved.


Frozen=false

In [ ]:
full_sequences = train_sequence.copy()

clean_sequences = train_sequence.copy()
if FORGETTEN_USER_ID in clean_sequences:
  del clean_sequences[FORGETTEN_USER_ID]
  print(f"Forgotten user {FORGETTEN_USER_ID}'s sequence is removed.")
else:
  print(f"Forgotten user {FORGETTEN_USER_ID} not found!!")

#training
baseline_model = train_engine("baseline_model", full_sequences, frozen_embeddings)
unlearned_model = train_engine("unlearned_model", clean_sequences, frozen_embeddings)

Forgotten user 4247's sequence is removed.


/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6044: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  warnings.warn(


Epoch: 1, Loss:  7.2912
Epoch: 2, Loss:  6.7664
Epoch: 3, Loss:  6.6256
Epoch: 4, Loss:  6.5364
Epoch: 5, Loss:  6.4729
Epoch: 6, Loss:  6.4205
Epoch: 7, Loss:  6.3786
Epoch: 8, Loss:  6.3462
Epoch: 9, Loss:  6.3198
Epoch: 10, Loss:  6.2976
baseline_model saved.
Epoch: 1, Loss:  7.3040
Epoch: 2, Loss:  6.7886
Epoch: 3, Loss:  6.6511
Epoch: 4, Loss:  6.5669
Epoch: 5, Loss:  6.4903
Epoch: 6, Loss:  6.4300
Epoch: 7, Loss:  6.3814
Epoch: 8, Loss:  6.3437
Epoch: 9, Loss:  6.3144
Epoch: 10, Loss:  6.2904
unlearned_model saved.
